============================================================
FinAgent — Main Pipeline
Course: IT Application in Banking and Finance
Run: python main.py
============================================================


In [ ]:

import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.figure_factory as ff
import seaborn as sns
import matplotlib.pyplot as plt
from groq import Groq



============================================================
============================================================


In [ ]:

GROQ_API_KEY = "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"
BASE_DIR     = os.getcwd()
INPUT_FILE   = os.path.join(BASE_DIR, "cleaned_data.xlsx")
OUTPUT_DIR   = BASE_DIR



============================================================
MODULE 1 — DATA CLEANING & PROCESSING
============================================================


In [ ]:

print("\n" + "="*60)
print("  MODULE 1 — DATA CLEANING & PROCESSING")
print("="*60)

stock = pd.read_csv(os.path.join(BASE_DIR, "stock2_prices.csv"))
macro = pd.read_csv(os.path.join(BASE_DIR, "macro2_prices.csv"))

stock = stock[stock.iloc[:, 0] != 'Date']
macro = macro[macro.iloc[:, 0] != 'Date']

stock.columns = [
    'Date',
    'AAPL_Close', 'AAPL_AdjClose', 'AAPL_Volume',
    'AMZN_Close', 'AMZN_AdjClose', 'AMZN_Volume',
    'JNJ_Close',  'JNJ_AdjClose',  'JNJ_Volume',
    'JPM_Close',  'JPM_AdjClose',  'JPM_Volume',
    'META_Close', 'META_AdjClose', 'META_Volume',
    'MSFT_Close', 'MSFT_AdjClose', 'MSFT_Volume',
    'NVDA_Close', 'NVDA_AdjClose', 'NVDA_Volume',
    'TSLA_Close', 'TSLA_AdjClose', 'TSLA_Volume',
]

macro.columns = [
    'Date',
    'SP500_Close', 'SP500_AdjClose', 'SP500_Volume',
    'Gold_Close',  'Gold_AdjClose',  'Gold_Volume',
    'Oil_Close',   'Oil_AdjClose',   'Oil_Volume',
]

stock['Date'] = pd.to_datetime(stock['Date'], errors='coerce', dayfirst=False)
macro['Date'] = pd.to_datetime(macro['Date'], errors='coerce', dayfirst=False)
stock.dropna(subset=['Date'], inplace=True)
macro.dropna(subset=['Date'], inplace=True)

for col in stock.columns[1:]:
    stock[col] = pd.to_numeric(stock[col], errors='coerce')
for col in macro.columns[1:]:
    macro[col] = pd.to_numeric(macro[col], errors='coerce')

data = pd.merge(stock, macro, on='Date', how='left')
data.sort_values('Date', inplace=True)
data.set_index('Date', inplace=True)

data.ffill(inplace=True)
data.bfill(inplace=True)
print(f"Missing values after fill: {data.isnull().sum().sum()}")

before = len(data)
data = data[~data.index.duplicated(keep='first')]
print(f"Removed {before - len(data)} duplicate rows")

close_cols = [
    'AAPL_Close', 'AMZN_Close', 'JNJ_Close', 'JPM_Close',
    'META_Close', 'MSFT_Close', 'NVDA_Close', 'TSLA_Close'
]

for col in close_cols:
    name = col.replace('_Close', '')
    data[f'{name}_return']     = data[col].pct_change()
    data[f'{name}_MA7']        = data[col].rolling(window=7).mean()
    data[f'{name}_MA30']       = data[col].rolling(window=30).mean()
    data[f'{name}_volatility'] = data[f'{name}_return'].rolling(window=30).std()

for asset in ['SP500', 'Gold', 'Oil']:
    col = f'{asset}_Close'
    data[f'{asset}_return']     = data[col].pct_change()
    data[f'{asset}_MA7']        = data[col].rolling(window=7).mean()
    data[f'{asset}_MA30']       = data[col].rolling(window=30).mean()
    data[f'{asset}_volatility'] = data[f'{asset}_return'].rolling(window=30).std()

data.dropna(inplace=True)

all_assets = [col.replace('_Close', '') for col in close_cols] + ['SP500', 'Gold', 'Oil']
for name in all_assets:
    ret_col = f'{name}_return'
    mean = data[ret_col].mean()
    std  = data[ret_col].std()
    data[f'{name}_is_outlier'] = (data[ret_col] - mean).abs() > 3 * std
    print(f"{name}: {data[f'{name}_is_outlier'].sum()} outliers detected")

data.reset_index(inplace=True)
data.to_excel(INPUT_FILE, index=False)
print(f"\n✅ Cleaned data saved: {INPUT_FILE}")



============================================================
MODULE 2 — VISUALIZATION
============================================================


In [ ]:

print("\n" + "="*60)
print("  MODULE 2 — VISUALIZATION")
print("="*60)

df = pd.read_excel(INPUT_FILE)
df['Date'] = pd.to_datetime(df['Date'])

# Chart 1: Price Trend + Volume
ticker = "AAPL"
fig1 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    subplot_titles=(f"{ticker} Price Trend", "Volume")
)
fig1.add_trace(go.Scatter(x=df['Date'], y=df[f'{ticker}_Close'], name='Close Price', line=dict(color='blue', width=1.5)), row=1, col=1)
fig1.add_trace(go.Scatter(x=df['Date'], y=df[f'{ticker}_MA7'], name='MA7', line=dict(color='orange', width=1, dash='dash')), row=1, col=1)
fig1.add_trace(go.Scatter(x=df['Date'], y=df[f'{ticker}_MA30'], name='MA30', line=dict(color='red', width=1, dash='dash')), row=1, col=1)
fig1.add_trace(go.Bar(x=df['Date'], y=df[f'{ticker}_Volume'], name='Volume', marker_color='lightblue'), row=2, col=1)
fig1.update_layout(title=f"{ticker} Price Trend with Volume Overlay (2023-2024)", height=600, template="plotly_white")
fig1.show()
print("✅ Chart 1 done")

# Chart 2: Correlation Heatmap
return_cols = [col for col in df.columns if col.endswith('_return')]
returns_df = df[return_cols].copy()
returns_df.columns = [col.replace('_return', '') for col in return_cols]
corr = returns_df.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap of Daily Returns (2023-2024)', fontsize=14)
plt.tight_layout()
plt.show()
print("✅ Chart 2 done")

# Chart 3: KDE Distribution
stocks_kde = ['NVDA', 'TSLA', 'JNJ', 'JPM']
colors_kde = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
data_kde = [df[f'{s}_return'].dropna().tolist() for s in stocks_kde]
fig3 = ff.create_distplot(data_kde, group_labels=stocks_kde, colors=colors_kde, show_hist=False, show_rug=False)
fig3.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.7)
fig3.update_layout(title='Distribution of Daily Returns - KDE (2023-2024)', xaxis_title='Daily Return', yaxis_title='Density', template='plotly_white', height=500)
fig3.show()
print("✅ Chart 3 done")

# Chart 4: Bollinger Bands
ticker4 = 'NVDA'
df['BB_std']   = df['NVDA_Close'].rolling(30).std()
df['BB_upper'] = df['NVDA_MA30'] + 2 * df['BB_std']
df['BB_lower'] = df['NVDA_MA30'] - 2 * df['BB_std']
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df['Date'], y=df['BB_upper'], name='BB Upper', line=dict(color='rgba(0,100,255,0.5)', width=1), fill=None))
fig4.add_trace(go.Scatter(x=df['Date'], y=df['BB_lower'], name='BB Lower', line=dict(color='rgba(0,100,255,0.5)', width=1), fill='tonexty', fillcolor='rgba(0,100,255,0.1)'))
fig4.add_trace(go.Scatter(x=df['Date'], y=df[f'{ticker4}_Close'], name='Close Price', line=dict(color='blue', width=1.5)))
fig4.add_trace(go.Scatter(x=df['Date'], y=df[f'{ticker4}_MA7'], name='MA7', line=dict(color='orange', width=1, dash='dash')))
fig4.add_trace(go.Scatter(x=df['Date'], y=df[f'{ticker4}_MA30'], name='MA30', line=dict(color='red', width=1, dash='dash')))
fig4.update_layout(title=f'{ticker4} Rolling Statistics with Bollinger Bands (2023-2024)', xaxis_title='Date', yaxis_title='Price (USD)', template='plotly_white', height=550)
fig4.show()
print("✅ Chart 4 done")



============================================================
MODULE 3 — AI ANALYSIS
============================================================


In [ ]:

print("\n" + "="*60)
print("  MODULE 3 — AI ANALYSIS")
print("="*60)

client  = Groq(api_key=GROQ_API_KEY)
TICKERS = ["AAPL","AMZN","JNJ","JPM","META","MSFT","NVDA","TSLA","SP500","Gold","Oil"]
COMPARE_PAIRS = [("NVDA","TSLA"),("AAPL","MSFT"),("Gold","SP500")]
SYSTEM = """You are a quantitative financial analyst assistant.
Be concise, professional, under 150 words per section.
Always reference specific numbers from the data provided.
Do NOT invent figures not in the data."""

def build_snapshot(ticker):
    close_col = f"{ticker}_Close"
    if close_col not in df.columns: return None
    cols = ["Date", close_col] + [
        f"{ticker}{s}" for s in ["_return","_MA7","_MA30","_volatility"]
        if f"{ticker}{s}" in df.columns
    ]
    recent = df[cols].dropna(subset=[close_col]).tail(30)
    if recent.empty: return None
    last, first = recent.iloc[-1], recent.iloc[0]
    return {
        "ticker":           ticker,
        "period":           f"{first['Date'].date()} to {last['Date'].date()}",
        "latest_close":     round(float(last[close_col]), 4),
        "30d_return_pct":   round((float(last[close_col])/float(first[close_col])-1)*100, 2),
        "latest_ma7":       round(float(last[f"{ticker}_MA7"]), 4)  if f"{ticker}_MA7"        in recent.columns else None,
        "latest_ma30":      round(float(last[f"{ticker}_MA30"]), 4) if f"{ticker}_MA30"       in recent.columns else None,
        "avg_volatility":   round(float(recent[f"{ticker}_volatility"].mean())*100, 4) if f"{ticker}_volatility" in recent.columns else None,
        "min_daily_return": round(float(recent[f"{ticker}_return"].min())*100, 4)      if f"{ticker}_return"    in recent.columns else None,
        "max_daily_return": round(float(recent[f"{ticker}_return"].max())*100, 4)      if f"{ticker}_return"    in recent.columns else None,
    }

def detect_anomalies():
    out = []
    for t in TICKERS:
        col = f"{t}_return"
        if col not in df.columns: continue
        s = df[col].dropna()
        mean, std = s.mean(), s.std()
        if std == 0: continue
        flagged = df[["Date",col]].dropna()
        flagged = flagged[(abs(flagged[col]-mean)/std) >= 3]
        for _, row in flagged.iterrows():
            out.append({"ticker":t, "date":str(row["Date"].date()),
                        "return_pct":round(float(row[col])*100,4),
                        "z_score":round((float(row[col])-mean)/std,2)})
    out.sort(key=lambda x: abs(x["z_score"]), reverse=True)
    return out[:15]

def ask_groq(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role":"system", "content":SYSTEM},
            {"role":"user",   "content":prompt}
        ],
        max_tokens=1024,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

snapshots = []
for t in TICKERS:
    s = build_snapshot(t)
    if s: snapshots.append(s)

results = {
    "generated_at":      datetime.now().isoformat(timespec="seconds"),
    "trend_summaries":   {},
    "anomaly_commentary":"",
    "risk_commentary":   "",
    "comparisons":       {}
}

print("[1/4] Trend summaries...", end=" ", flush=True)
raw = ask_groq(
    f"For each of the {len(snapshots)} assets below, write a 2-3 sentence trend summary "
    f"referencing MA7 vs MA30 relationship and the 30-day return. "
    f"Respond ONLY as valid JSON with ticker as key and summary as value, no extra text.\n\n"
    f"Data:\n{json.dumps(snapshots, indent=2)}"
)
try:
    clean = raw[raw.index("{"):raw.rindex("}")+1]
    results["trend_summaries"] = json.loads(clean)
    print("✅")
except:
    results["trend_summaries"] = {s["ticker"]: "See raw output" for s in snapshots}
    print("⚠ JSON parse issue")

print("[2/4] Anomaly detection...", end=" ", flush=True)
anomalies = detect_anomalies()
results["anomaly_commentary"] = ask_groq(
    f"Write a 4-6 sentence paragraph about the most significant return anomalies. "
    f"Mention specific tickers, dates, and z-scores.\n\nData:\n{json.dumps(anomalies, indent=2)}"
) if anomalies else "No significant anomalies detected."
print(f"✅ ({len(anomalies)} anomalies)")

print("[3/4] Risk commentary...", end=" ", flush=True)
risk_data = [
    {"ticker":s["ticker"], "avg_volatility%":s["avg_volatility"],
     "worst_day%":s["min_daily_return"], "best_day%":s["max_daily_return"]}
    for s in snapshots
]
results["risk_commentary"] = ask_groq(
    f"Rank all {len(risk_data)} assets from highest to lowest risk. "
    f"Write a 5-6 sentence risk commentary referencing specific volatility percentages.\n\n"
    f"Data:\n{json.dumps(risk_data, indent=2)}"
)
print("✅")

print("[4/4] Comparisons...", end=" ", flush=True)
pairs_data = []
for (a, b) in COMPARE_PAIRS:
    sa = next((s for s in snapshots if s["ticker"]==a), None)
    sb = next((s for s in snapshots if s["ticker"]==b), None)
    if sa and sb:
        pairs_data.append({"pair": f"{a} vs {b}", "A": sa, "B": sb})
raw_pairs = ask_groq(
    f"For each pair below, write a 3-4 sentence comparison covering return performance, "
    f"volatility difference, trend direction, and a risk-adjusted observation. "
    f"Respond ONLY as valid JSON with pair name as key, no extra text.\n\n"
    f"Data:\n{json.dumps(pairs_data, indent=2)}"
)
try:
    clean = raw_pairs[raw_pairs.index("{"):raw_pairs.rindex("}")+1]
    results["comparisons"] = json.loads(clean)
    print("✅")
except:
    results["comparisons"] = {f"{a} vs {b}": raw_pairs for a,b in COMPARE_PAIRS}
    print("⚠ JSON parse issue")

sep = "=" * 65
print(f"\n\n{sep}")
print("  FINAGENT – AI ANALYSIS REPORT (Groq LLaMA 3.3 70B)")
print(f"  Generated : {results['generated_at']}")
print(f"  Assets    : {', '.join(TICKERS)}")
print(sep)

print("\n📈  SECTION 1 — TREND SUMMARIES")
print("-" * 65)
for ticker, text in results["trend_summaries"].items():
    print(f"\n▶ {ticker}")
    print(text)

print("\n\n⚠   SECTION 2 — ANOMALY COMMENTARY")
print("-" * 65)
print(results["anomaly_commentary"])

print("\n\n🔴  SECTION 3 — RISK COMMENTARY")
print("-" * 65)
print(results["risk_commentary"])

print("\n\n⚖   SECTION 4 — ASSET COMPARISONS")
print("-" * 65)
for label, text in results["comparisons"].items():
    print(f"\n▶ {label}")
    print(text)

print(f"\n{sep}")

with open(os.path.join(OUTPUT_DIR, "ai_analysis_output.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("\n💾 Saved: ai_analysis_output.json")

print("\n" + "="*60)
print("  FINAGENT PIPELINE COMPLETED SUCCESSFULLY")
print("="*60)
